CODE DUMMY DATA GENERATOR INI HANYA UNTUK SEKALI RUN AJA, KARENA MENYESUAIKAN STUDY CASE DIMANA KE 3 TABLE (CUSTOMERS_RAW, SALES, AFTER_SALES) SUDAH EXSITING DI DATABASE MYSQL

Generate Data Tabel customers_raw

In [16]:
import random
from faker import Faker
from datetime import datetime

fake = Faker()

def random_dob():
    formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d/%m/%Y",
        None
    ]

    chosen_format = random.choice(formats)

    if chosen_format is None:
        return None

    date = fake.date_of_birth(minimum_age=15, maximum_age=100)
    return date.strftime(chosen_format)

def random_name():
    if random.random() < 0.2:
        return fake.company()
    else:
        return fake.first_name()

def random_created_at():
    dt = fake.date_time_this_year()

    # generate millisecond (0–999)
    ms = random.randint(0, 999)

    # format ke string + millisecond
    return dt.strftime('%Y-%m-%d %H:%M:%S') + f'.{ms:03d}'

def generate_data(n=1000):
    data = []

    for i in range(7, n+1):
        name = random_name()
        dob = random_dob()
        created_at = random_created_at()

        data.append((i, name, dob, created_at))

    return data


# Example output
data = generate_data(1000)

for row in data:
    print(row)

(7, 'Johnson, Brock and Padilla', '27/10/1973', '2026-01-16 02:23:14.616')
(8, 'Lorraine', '2004-05-15', '2026-01-31 04:38:24.517')
(9, 'Rivera-Anderson', '2008-08-09', '2026-03-09 12:14:04.366')
(10, 'Julie', '12/09/1980', '2026-03-04 13:46:25.242')
(11, 'Christine', None, '2026-01-15 18:23:33.655')
(12, 'Morris, Farmer and Adams', '29/10/1961', '2026-03-06 05:29:12.539')
(13, 'Shelia', None, '2026-01-19 20:49:28.963')
(14, 'Cervantes, Carter and Guerra', None, '2026-01-06 14:28:54.778')
(15, 'Katelyn', '1984-04-09', '2026-03-26 03:40:01.532')
(16, 'Amy', '1961/11/20', '2026-01-28 11:45:37.209')
(17, 'Kelli', '2010/05/22', '2026-03-05 04:06:57.269')
(18, 'Logan, Bailey and Cook', '1981/12/01', '2026-03-05 05:50:14.837')
(19, 'Dawn', '28/10/1971', '2026-02-02 17:44:05.735')
(20, 'Cynthia', None, '2026-02-09 14:28:08.288')
(21, 'Darryl', '1940-12-16', '2026-02-20 08:38:32.802')
(22, 'Jennings PLC', '1982/05/26', '2026-02-25 15:25:45.527')
(23, 'Dillon', None, '2026-03-17 09:04:43.604')


INJECT TO TABLE STG_CUSTOMER_RAW

In [ ]:
import mysql.connector
from getpass import getpass

password = getpass("Masukkan password MySQL: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="mysql-dwh-dev"
)

cursor = conn.cursor()

# data = generate_data(1000)

cursor.executemany("""
    INSERT INTO STG_CUSTOMERS_RAW (id, name, dob, created_at)
    VALUES (%s, %s, %s, %s)
""", data)

conn.commit()
cursor.close()
conn.close()

DATA TABLE SALES_RAW

In [ ]:
import random
from datetime import datetime, timedelta

# SAMPLE MASTER DATA
model_price_map = {
    'RAIZA': '350.000.000',
    'RANGGO': '430.000.000',
    'INNAVO': '600.000.000',
    'VELOS': '390.000.000'
}

models = list(model_price_map.keys())

# SIMULASI customer created_at
def generate_customer_master(n=1000):
    data = {}
    base_date = datetime(2025, 1, 1)
    for i in range(1, n+1):
        random_days = random.randint(0, 60)
        random_seconds = random.randint(0, 86400)
        dt = base_date + timedelta(days=random_days, seconds=random_seconds)
        data[i] = dt
    return data

customer_created_at = generate_customer_master(1000)

# GENERATE VIN UNIQUE
used_vins = set()
def generate_unique_vin():
    while True:
        vin = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', k=10))
        if vin not in used_vins:
            used_vins.add(vin)
            return vin

# Generate data
def generate_data(n=2000):
    data = []
    for _ in range(n):
        customer_id = random.randint(1, 1000)
        cust_dt = customer_created_at[customer_id]

        # Gunakan satu datetime untuk invoice_date & created_at
        invoice_created_dt = cust_dt + timedelta(days=random.randint(0, 30), seconds=random.randint(0, 86400))

        vin = generate_unique_vin()
        model = random.choice(models)
        price = model_price_map[model]

        created_at = invoice_created_dt.strftime('%Y-%m-%d %H:%M:%S') + f'.{int(invoice_created_dt.microsecond/1000):03d}'
        invoice_date = invoice_created_dt.strftime('%Y-%m-%d')

        data.append((
            vin,
            customer_id,
            model,
            invoice_date,
            price,
            created_at
        ))

    return data

# GENERATE DATA
data = generate_data()

# PRINT SAMPLE
for row in data[:10]:
    print(row)

('5SV5H0EJWQ', 17, 'INNAVO', '2025-02-18', '600.000.000', '2025-02-18 18:26:07.000')
('AX47YKVAXB', 869, 'RAIZA', '2025-01-27', '350.000.000', '2025-01-27 11:21:52.000')
('Y0GFYJL31T', 848, 'RANGGO', '2025-03-02', '430.000.000', '2025-03-02 20:51:14.000')
('EVX5V9151W', 471, 'RANGGO', '2025-01-12', '430.000.000', '2025-01-12 04:27:40.000')
('8MYWTURKY7', 687, 'INNAVO', '2025-01-15', '600.000.000', '2025-01-15 18:09:09.000')
('ES2JFL3W0I', 111, 'RAIZA', '2025-02-08', '350.000.000', '2025-02-08 12:40:33.000')
('7WTCQBM9D4', 694, 'RAIZA', '2025-03-17', '350.000.000', '2025-03-17 08:14:55.000')
('Y5GRME8U0Z', 858, 'RANGGO', '2025-02-11', '430.000.000', '2025-02-11 08:55:28.000')
('TRX3BCIOD2', 437, 'VELOS', '2025-03-28', '390.000.000', '2025-03-28 14:36:52.000')
('CTF2OBTUSZ', 945, 'INNAVO', '2025-02-19', '600.000.000', '2025-02-19 12:26:08.000')


INJECT TO TABLE SALES_RAW

In [5]:
import mysql.connector
from getpass import getpass

password = getpass("Masukkan password MySQL: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="mysql-dwh-dev"
)

cursor = conn.cursor()

# data = generate_data(1000)

cursor.executemany("""
    INSERT INTO STG_SALES_RAW (
    vin,
    customer_id,
    model,
    invoice_date,
    price,
    created_at)
    VALUES (%s, %s, %s, %s, %s, %s)
""", data)

conn.commit()
cursor.close()
conn.close()

DUMMY DATA AFTER SALES RAW

In [6]:
import random
import string
from datetime import datetime, timedelta

# GENERATOR VIN UNIK
used_vins = set()
def generate_unique_vin():
    while True:
        vin = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', k=10))
        if vin not in used_vins:
            used_vins.add(vin)
            return vin

# MASTER DATA
model_map = {
    'JIS8135SAD': 'RAIZA',
    'MAS8160POE': 'RANGGO',
    'JLK1368KDE': 'INNAVO',
    'JLK1869KDF': 'VELOS',
    'JLK1962KOP': 'VELOS'
}
service_types = ['BP', 'PM', 'GR']

# SIMULASI SALES_RAW created_at
def generate_sales_master(n=1000):
    data = {}
    base_date = datetime(2025, 1, 1)
    for i in range(1, n+1):
        dt = base_date + timedelta(
            days=random.randint(0, 180),
            seconds=random.randint(0, 86400)
        )
        data[i] = dt
    return data

sales_created_at = generate_sales_master(1000)

# GENERATE SERVICE TICKET
def generate_ticket():
    number = random.randint(100, 999)
    letters = ''.join(random.choices(string.ascii_lowercase, k=3))
    suffix = random.randint(1, 9)
    return f"T{number}-{letters}{suffix}"

# GENERATE DATETIME AFTER SALES CREATED_AT
def random_datetime_after(base_dt):
    max_date = datetime(2026, 12, 31)
    delta = max_date - base_dt
    random_seconds = random.randint(1, int(delta.total_seconds()))
    new_dt = base_dt + timedelta(seconds=random_seconds)
    ms = random.randint(0, 999)
    new_dt = new_dt.replace(microsecond=ms * 1000)
    return new_dt

# GENERATE DATA
def generate_data(n=500):
    data = []

    for _ in range(n):
        customer_id = random.randint(1, 1000)
        sales_dt = sales_created_at[customer_id]

        created_at_dt = random_datetime_after(sales_dt)
        service_date = created_at_dt.strftime('%Y-%m-%d')
        created_at = created_at_dt.strftime('%Y-%m-%d %H:%M:%S') + f'.{int(created_at_dt.microsecond/1000):03d}'

        vin = generate_unique_vin()
        # assign random model untuk VIN baru
        model = random.choice(['RAIZA', 'RANGGO', 'INNAVO', 'VELOS'])
        model_map[vin] = model  # simpan mapping VIN -> model
        service_type = random.choice(service_types)
        ticket = generate_ticket()

        data.append((
            ticket,
            vin,
            customer_id,
            model,
            service_date,
            service_type,
            created_at
        ))

    return data

data = generate_data(1000)

for row in data:
    print(row)

('T130-osm8', 'D4L0Y0WR2B', 217, 'RANGGO', '2025-12-30', 'GR', '2025-12-30 12:21:42.775')
('T205-npn2', '4PZ8PKH3MU', 711, 'RANGGO', '2025-06-23', 'BP', '2025-06-23 20:55:26.732')
('T806-ahy9', 'ZLDH9WU52O', 665, 'RANGGO', '2026-01-02', 'GR', '2026-01-02 03:44:09.886')
('T771-nci2', '2YSB114TLJ', 703, 'INNAVO', '2026-11-30', 'GR', '2026-11-30 21:20:32.355')
('T360-tfk3', 'F1GBIHBF83', 485, 'RAIZA', '2026-10-02', 'GR', '2026-10-02 17:36:29.994')
('T463-wkw9', 'HLZPE3PB0B', 858, 'VELOS', '2026-11-08', 'PM', '2026-11-08 06:34:16.298')
('T252-jjw3', 'HPDHCKR1VD', 160, 'RAIZA', '2026-03-19', 'GR', '2026-03-19 06:25:21.030')
('T843-iuo4', '7FD6X8EX5W', 226, 'VELOS', '2025-12-12', 'GR', '2025-12-12 19:47:36.120')
('T316-mso1', 'XYMKLWBM7E', 743, 'RAIZA', '2026-11-17', 'PM', '2026-11-17 12:37:48.458')
('T568-ojh8', 'VN7NZ3Z8RE', 339, 'RAIZA', '2026-04-21', 'GR', '2026-04-21 05:29:45.453')
('T928-fdd9', 'NDIR6YLQ5J', 316, 'VELOS', '2026-04-23', 'GR', '2026-04-23 10:49:40.676')
('T460-tvp2', '0V

In [21]:
import random
import string
from datetime import datetime, timedelta

# MASTER DATA
vin_list = ['JIS8135SAD', 'MAS8160POE', 'JLK1368KDE', 'JLK1869KDF', 'JLK1962KOP']

model_map = {
    'JIS8135SAD': 'RAIZA',
    'MAS8160POE': 'RANGGO',
    'JLK1368KDE': 'INNAVO',
    'JLK1869KDF': 'VELOS',
    'JLK1962KOP': 'VELOS'
}

service_types = ['BP', 'PM', 'GR']


# SIMULASI SALES_RAW created_at
def generate_sales_master(n=1000):
    data = {}
    base_date = datetime(2025, 1, 1)

    for i in range(1, n+1):
        dt = base_date + timedelta(
            days=random.randint(0, 180),
            seconds=random.randint(0, 86400)
        )
        data[i] = dt

    return data

sales_created_at = generate_sales_master(1000)


# GENERATE SERVICE TICKET
def generate_ticket():
    number = random.randint(100, 999)
    letters = ''.join(random.choices(string.ascii_lowercase, k=3))
    suffix = random.randint(1, 9)
    return f"T{number}-{letters}{suffix}"


# GENERATE DATETIME AFTER SALES CREATED_AT
def random_datetime_after(base_dt):
    max_date = datetime(2026, 12, 31)

    delta = max_date - base_dt
    random_seconds = random.randint(1, int(delta.total_seconds()))

    new_dt = base_dt + timedelta(seconds=random_seconds)

    # millisecond
    ms = random.randint(0, 999)
    new_dt = new_dt.replace(microsecond=ms * 1000)

    return new_dt


def generate_data(n=1000):
    data = []

    for _ in range(n):
        customer_id = random.randint(1, 1000)

        # ambil created_at dari sales_raw
        sales_dt = sales_created_at[customer_id]

        # generate service datetime (HARUS setelah sales)
        created_at_dt = random_datetime_after(sales_dt)

        service_date = created_at_dt.strftime('%Y-%m-%d')

        created_at = created_at_dt.strftime('%Y-%m-%d %H:%M:%S') + f'.{int(created_at_dt.microsecond/1000):03d}'

        vin = random.choice(vin_list)
        model = model_map[vin]
        service_type = random.choice(service_types)
        ticket = generate_ticket()

        data.append((
            ticket,
            vin,
            customer_id,
            model,
            service_date,
            service_type,
            created_at
        ))

    return data


# GENERATE
data = generate_data(500)

# PRINT
for row in data:
    print(row)

('T264-wuy8', 'JIS8135SAD', 285, 'RAIZA', '2026-08-29', 'PM', '2026-08-29 22:30:12.537')
('T963-lxa5', 'JIS8135SAD', 930, 'RAIZA', '2026-06-03', 'BP', '2026-06-03 11:52:12.330')
('T967-xyv6', 'JLK1962KOP', 243, 'VELOS', '2025-12-03', 'GR', '2025-12-03 13:40:36.509')
('T371-cds8', 'JIS8135SAD', 404, 'RAIZA', '2026-03-20', 'GR', '2026-03-20 12:18:40.285')
('T862-nga7', 'JLK1368KDE', 259, 'INNAVO', '2025-05-11', 'BP', '2025-05-11 22:44:42.514')
('T685-ppr6', 'JLK1869KDF', 147, 'VELOS', '2025-11-02', 'BP', '2025-11-02 15:50:18.554')
('T301-vlm7', 'JLK1869KDF', 936, 'VELOS', '2025-07-27', 'PM', '2025-07-27 17:50:55.374')
('T265-thn1', 'JLK1368KDE', 23, 'INNAVO', '2026-01-10', 'GR', '2026-01-10 04:04:21.090')
('T954-eyo7', 'JLK1368KDE', 701, 'INNAVO', '2026-01-18', 'BP', '2026-01-18 10:09:34.201')
('T493-sge5', 'JLK1869KDF', 805, 'VELOS', '2026-08-19', 'PM', '2026-08-19 12:43:54.959')
('T236-uos1', 'JLK1368KDE', 709, 'INNAVO', '2026-01-08', 'GR', '2026-01-08 17:13:27.684')
('T540-rem7', 'MAS

INJECT TO TABLE STG_AFTER_SALES_RAW

In [8]:
import mysql.connector
from getpass import getpass

password = getpass("Masukkan password MySQL: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="mysql-dwh-dev"
)

cursor = conn.cursor()

# data = generate_data(1000)

cursor.executemany("""
    INSERT INTO STG_AFTER_SALES_RAW (
    service_ticket,
    vin,
    customer_id,
    model,
    service_date,
    service_type,
    created_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", data)

conn.commit()
cursor.close()
conn.close()

DATA TABLE CUSTOMER_ADDRESS

In [ ]:
import random
from faker import Faker
from datetime import datetime, timedelta
import os

fake = Faker('id_ID')

DELIMITER = "|@~"

# 📅 ambil tanggal hari ini
today_str = datetime.now().strftime('%Y%m%d')

# 📂 path dengan nama file harian
file_path = fr"D:\Git\Repository\End-To-End-Data-Pipeline-Using-Airflow-Docker-MySQL-CSV\data\customer_address_{today_str}.csv"


# SIMULASI customers_raw
def generate_customer_master(n=1000):
    data = {}
    base_date = datetime(2026, 1, 1)

    for i in range(1, n+1):
        dt = base_date + timedelta(
            days=random.randint(0, 90),
            seconds=random.randint(0, 86400)
        )
        ms = random.randint(0, 999)
        dt = dt.replace(microsecond=ms * 1000)
        data[i] = dt

    return data


customer_created_at = generate_customer_master(1000)


def generate_data(n=1000):
    if n > 1000:
        raise ValueError("Max 1000 rows")

    data = []
    customer_ids = random.sample(range(1, 1001), n)

    for i, cust_id in enumerate(customer_ids, start=1):
        created_at_dt = customer_created_at[cust_id]

        created_at = created_at_dt.strftime('%Y-%m-%d %H:%M:%S') + \
                     f'.{int(created_at_dt.microsecond/1000):03d}'

        address = fake.street_address().replace('\n', ' ')
        city = fake.city()
        province = fake.state()

        data.append([
            str(i),
            str(cust_id),
            address,
            city,
            province,
            created_at
        ])

    return data


# GENERATE
data = generate_data(1000)

# SAVE FILE
os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'w', encoding='utf-8') as f:
    # header
    f.write(DELIMITER.join(["id","customer_id","address","city","province","created_at"]) + "\n")

    # rows
    for row in data:
        f.write(DELIMITER.join(row) + "\n")

print(f"File berhasil dibuat: {file_path}")

File berhasil dibuat: D:\Technical Test - AstraWorld\customer_address_20260328.csv
